# 01r finite PEPS convergence benchmark

这个 notebook 用来验证 `backend="peps"` 是否真正作为三能级 `01r` finite PEPS 工作。

物理模型是每个格点有 `|0>`, `|1>`, `|r>` 三个局域能级的 Rydberg lattice。局域哈密顿量包含 `|0><1| + h.c.` hyperfine drive、`|1><r| + h.c.` Rydberg drive、`n_1`/`n_r` detuning；相互作用项是 `V_ij n_i^r n_j^r`。PEPS 的物理 leg 维度应为 `d=3`，而不是把 `|0>` 态丢掉后的有效二能级模型。

预期结果：小的 2x2 系统应该和 exact evolution 一致；随着 `dt` 减小、`chi_max` 增大，PEPS 的 `n_r` 曲线和逐格点 occupation 应向 exact 收敛。truncation error 用来判断 bond truncation 是否主导误差。


In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

import ryd_gate as rg
from ryd_gate import RydbergSystem
from ryd_gate.core.level_structures import InteractionSpec
from ryd_gate.lattice import Register
from ryd_gate.protocols.digital_analog import DigitalAnalogProtocol

plt.rcParams.update({"figure.dpi": 120})

## 1. 构造 01r 二维 lattice 和 time-dependent protocol

这里用 2x2 是为了保留 exact baseline。`Omega_R` 驱动 `|1> <-> |r>`，`Omega_hf` 驱动 `|0> <-> |1>`，两个 detuning 分别作用在 `n_r` 和 `n_1`。相互作用只取最近邻，方便 PEPS gate 分层，也方便和 exact 对照。

物理意义：如果 PEPS backend 真的是三能级，`Omega_hf` 会把 population 从 `|1>` 转到 `|0>`；如果它仍是二能级 backend，这一项会被忽略或直接报错。

In [ ]:
Lx, Ly = 2, 2
V_nn = 0.8
Omega_R = 1.0
Omega_hf = 0.35
Delta_R = 0.15
Delta_hf = -0.08
t_gate = 0.8

geom = Register.rectangle(Lx, Ly, spacing_um=1.0)
interaction = InteractionSpec(C6=V_nn, mode="nn")
protocol = DigitalAnalogProtocol.constant(
    omega_R=Omega_R,
    omega_hf=Omega_hf,
    delta_R=Delta_R,
    delta_hf=Delta_hf,
    t_gate=t_gate,
    n_steps=240,
)
system = RydbergSystem.from_lattice(
    geom,
    "01r",
    interaction=interaction,
    protocol=protocol,
)

initial_state = "all_1"
# 选择 0.2/Omega_R 的整数倍；这样 0.2、0.1、0.05 三组 dt 都会精确命中同一批 t_eval。
t_eval = np.linspace(0.0, t_gate, 6)

coords = np.asarray(geom.coords)
x_vals = np.unique(coords[:, 0])
y_vals = np.unique(coords[:, 1])

print("N =", system.N, "local_dim =", system.basis.local_dim, "Hilbert dim =", system.dim)
print("interaction pairs =", system.meta("interaction_pairs"))
print("t_eval =", t_eval)

## 2. Exact baseline

Exact evolution 在 `3^4 = 81` 维 Hilbert space 中直接演化态矢量。它不适合大 lattice，但在这里是 PEPS convergence 的基准。我们记录 `n_0`, `n_1`, `n_r` 和平均 Rydberg occupation。

In [ ]:
tic = time.perf_counter()
exact_res = rg.simulate(system, [], initial_state, backend="exact", t_eval=t_eval)
exact_time = time.perf_counter() - tic

exact_n0 = np.asarray([
    [system.expectation(f"n_0_{i}", psi) for i in range(system.N)]
    for psi in exact_res.states
])
exact_n1 = np.asarray([
    [system.expectation(f"n_1_{i}", psi) for i in range(system.N)]
    for psi in exact_res.states
])
exact_nr = np.asarray([
    [system.expectation(f"n_r_{i}", psi) for i in range(system.N)]
    for psi in exact_res.states
])
exact_n_mean = exact_nr.mean(axis=1)

print(f"exact runtime: {exact_time:.3f} s")
print("time      <n_r> exact")
for t, val in zip(t_eval, exact_n_mean):
    print(f"{t:7.4f}   {val:10.6f}")

fig, ax = plt.subplots(figsize=(5.5, 3.2))
ax.plot(t_eval, exact_n0.mean(axis=1), "o-", label=r"exact $\langle n_0\rangle")
ax.plot(t_eval, exact_n1.mean(axis=1), "o-", label=r"exact $\langle n_1\rangle")
ax.plot(t_eval, exact_n_mean, "o-", label=r"exact $\langle n_r\rangle")
ax.set_xlabel("time")
ax.set_ylabel("mean population")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

## 3. PEPS convergence sweep

这个 block 直接扫 `dt = 0.2/Omega_R, 0.1/Omega_R, 0.05/Omega_R`，`chi_max = 4, 6, 8, 12`，以及 `update_environment = NTU/CTM`。默认 measurement 使用 BP，因为它快；后面会对最佳参数单独用 CTM measurement 复查。

预期：`dt` 越小 Trotter error 越小，`chi_max` 越大 truncation error 越小。若某组参数不收敛，通常会表现为 `max_site_error` 不随 `dt`/`chi` 单调下降，或者 truncation error 明显偏大。

In [ ]:
dt_factors = [0.2, 0.1, 0.05]
chi_values = [4, 6, 8, 12]
update_envs = ["ntu", "ctm"]

rows = []
peps_results = {}

for update_env in update_envs:
    for chi in chi_values:
        for factor in dt_factors:
            dt = factor / Omega_R
            key = (update_env, chi, factor)
            tic = time.perf_counter()
            try:
                res = rg.simulate(
                    system,
                    [],
                    initial_state,
                    backend="peps",
                    t_eval=t_eval,
                    observables=["n_0", "n_1", "n_r", "n_mean"],
                    backend_options={
                        "chi_max": chi,
                        "dt": dt,
                        "svd_min": 1e-10,
                        "update_environment": update_env,
                        "measurement_environment": "bp",
                        "initialization": "SVD",
                        "max_iter": 30,
                        "tol_iter": 1e-12,
                        "use_cuda": False,
                    },
                )
                runtime = time.perf_counter() - tic
                nr = np.asarray(res.metadata["obs"]["n_r"])
                nmean = np.asarray(res.metadata["obs"]["n_mean"])
                max_mean_error = float(np.max(np.abs(nmean - exact_n_mean)))
                max_site_error = float(np.max(np.abs(nr - exact_nr)))
                max_trunc = float(res.metadata.get("max_truncation_error", np.nan))
                acc_trunc = res.metadata.get("accumulated_truncation_error")
                rows.append({
                    "env": update_env,
                    "chi": chi,
                    "dt_factor": factor,
                    "status": "ok",
                    "time_s": runtime,
                    "max_mean_error": max_mean_error,
                    "max_site_error": max_site_error,
                    "max_truncation_error": max_trunc,
                    "accumulated_truncation_error": acc_trunc,
                })
                peps_results[key] = res
                print(f"ok   env={update_env:3s} chi={chi:2d} dt={factor:.2f}/Omega  "
                      f"time={runtime:7.2f}s site_err={max_site_error:.3e} trunc={max_trunc:.3e}")
            except Exception as exc:
                runtime = time.perf_counter() - tic
                rows.append({
                    "env": update_env,
                    "chi": chi,
                    "dt_factor": factor,
                    "status": exc.__class__.__name__,
                    "time_s": runtime,
                    "max_mean_error": np.nan,
                    "max_site_error": np.nan,
                    "max_truncation_error": np.nan,
                    "accumulated_truncation_error": np.nan,
                })
                print(f"fail env={update_env:3s} chi={chi:2d} dt={factor:.2f}/Omega  "
                      f"time={runtime:7.2f}s {exc.__class__.__name__}: {exc}")

try:
    import pandas as pd
    conv_df = pd.DataFrame(rows)
    display(conv_df)
except ImportError:
    conv_df = rows
    for row in rows:
        print(row)

## 4. 选择最佳 PEPS 参数并画平均 occupation

这里自动从成功的 sweep 结果里选 `max_site_error` 最小的一组。图中 exact 和 PEPS 的平均 `n_r` 曲线应该重合；如果不重合，优先看 truncation error 和 `dt` 是否还不够小。

In [ ]:
ok_rows = [row for row in rows if row["status"] == "ok"]
if not ok_rows:
    raise RuntimeError("No PEPS convergence run completed. Check that yastn is installed and backend options are valid.")

best = min(ok_rows, key=lambda row: row["max_site_error"])
best_key = (best["env"], best["chi"], best["dt_factor"])
best_res = peps_results[best_key]
best_nr = np.asarray(best_res.metadata["obs"]["n_r"])
best_nmean = np.asarray(best_res.metadata["obs"]["n_mean"])

print("best PEPS row:")
print(best)

fig, ax = plt.subplots(figsize=(5.6, 3.2))
ax.plot(t_eval, exact_n_mean, "o-", label="exact")
ax.plot(best_res.times, best_nmean, "s--", label=f"PEPS {best_key}")
ax.set_xlabel("time")
ax.set_ylabel(r"$\langle n_r \rangle$")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

trunc = np.asarray(best_res.metadata.get("truncation_error", []), dtype=float)
if trunc.size:
    fig_t, ax_t = plt.subplots(figsize=(5.6, 3.0))
    ax_t.plot(np.arange(1, trunc.size + 1), trunc, ".-")
    ax_t.set_xlabel("PEPS time step")
    ax_t.set_ylabel("max truncation error in step")
    ax_t.set_yscale("symlog", linthresh=1e-14)
    ax_t.grid(alpha=0.3)
    plt.show()

## 5. 逐格点 Rydberg occupation color map

这个图检查 PEPS 不只是平均值正确，空间分布也要和 exact 一致。对有 local addressing 或非均匀 protocol 的情况，这个 block 比平均值更重要。

In [ ]:
fig_h, axes_h = plt.subplots(2, len(t_eval), figsize=(2.25 * len(t_eval), 4.8), constrained_layout=True)

for row_idx, (label, occ_series) in enumerate([("exact", exact_nr), ("PEPS", best_nr)]):
    for ax_h, t, occ in zip(axes_h[row_idx], t_eval, occ_series):
        grid = occ.reshape(Lx, Ly)
        im = ax_h.imshow(grid.T, origin="lower", vmin=0.0, vmax=1.0, cmap="viridis")
        ax_h.set_title(f"{label}\nt={t:.2f}")
        ax_h.set_xticks(range(Lx))
        ax_h.set_yticks(range(Ly))
        if ax_h is axes_h[row_idx, 0]:
            ax_h.set_ylabel("y")
        else:
            ax_h.set_yticklabels([])
        ax_h.set_xlabel("x")
        for ix in range(Lx):
            for iy in range(Ly):
                val = grid[ix, iy]
                txt_color = "white" if val > 0.55 else "black"
                ax_h.text(ix, iy, f"{val:.2f}", ha="center", va="center", color=txt_color, fontsize=8)

fig_h.colorbar(im, ax=axes_h.ravel().tolist(), shrink=0.78, label=r"$\langle n_r \rangle$")
plt.show()

## 6. 用 CTM measurement 复查最佳参数

前面的 sweep 用 BP measurement 是为了快速筛参数。这个 block 对最佳 `dt/chi/update_env` 重跑一次，并把 measurement environment 改成 CTM。若 CTM 和 BP 的 `n_r` 差异很小，说明 observable contraction 不是主要误差源；若差异明显，应优先增大 measurement bond/environment 精度，而不是只增大 PEPS `chi_max`。

In [ ]:
tic = time.perf_counter()
ctm_res = rg.simulate(
    system,
    [],
    initial_state,
    backend="peps",
    t_eval=t_eval,
    observables=["n_r", "n_mean"],
    backend_options={
        "chi_max": int(best["chi"]),
        "dt": float(best["dt_factor"]) / Omega_R,
        "svd_min": 1e-10,
        "update_environment": best["env"],
        "measurement_environment": "ctm",
        "initialization": "SVD",
        "max_iter": 30,
        "tol_iter": 1e-12,
        "use_cuda": False,
    },
)
ctm_time = time.perf_counter() - tic
ctm_nr = np.asarray(ctm_res.metadata["obs"]["n_r"])
ctm_nmean = np.asarray(ctm_res.metadata["obs"]["n_mean"])

print(f"CTM measurement runtime: {ctm_time:.2f} s")
print("max |BP-CTM| per-site n_r:", np.max(np.abs(best_nr - ctm_nr)))
print("max |CTM-exact| per-site n_r:", np.max(np.abs(ctm_nr - exact_nr)))

fig, ax = plt.subplots(figsize=(5.6, 3.2))
ax.plot(t_eval, exact_n_mean, "o-", label="exact")
ax.plot(best_res.times, best_nmean, "s--", label="PEPS BP measurement")
ax.plot(ctm_res.times, ctm_nmean, "d:", label="PEPS CTM measurement")
ax.set_xlabel("time")
ax.set_ylabel(r"$\langle n_r \rangle$")
ax.grid(alpha=0.3)
ax.legend()
plt.show()